In [0]:
# %pip install geopandas
# dbutils.library.restartPython()

In [0]:
# =============================================================================
# jobs/06_compute_lookup.py
# Compute the static EA flood area / parliamentary constituency lookup table.
#
# What this script does:
#   1. Loads EA flood areas and constituency boundaries from Delta.
#   2. Runs a spatial join using GeoPandas to find all intersections.
#   3. Records the intersection percentage for reference (no minimum threshold).
#   4. Overwrites ea_area_constituency_lookup in full.
#
# Run triggers:
#   - After 03_refresh_ea_flood_areas (weekly)
#   - After 05_refresh_boundaries (quarterly / post-election)
#
# WHY GEOPANDAS ON THE DRIVER:
# GeoPandas loads all geometries into memory on the Spark driver node and
# runs the spatial join there. This is appropriate here because:
#   - EA flood areas: a few thousand polygons
#   - Constituencies: 650 polygons
# These fit easily in memory. Distributed spatial joins add complexity
# without benefit at this scale.
# =============================================================================

import sys
import json

sys.path.insert(0, "/Workspace/Users/jon.payne@environment-agency.gov.uk/FGS_Notebooks/")
from config import (
    TBL_EA_FWA, TBL_EA_FAA, TBL_CONSTITUENCIES, TBL_EA_CONST_LOOKUP
)
from utils.helpers import get_spark, utc_now
from pyspark.sql.types import StructType, StructField, StringType, FloatType

import geopandas as gpd
import pandas as pd
from shapely.geometry import shape
from pyspark.sql import Row


# =============================================================================
# SETUP
# =============================================================================

spark   = get_spark()
now_iso = utc_now().isoformat()

In [0]:
%skip
# =============================================================================
# GEOMETRY CLEANING
# buffer(0) repairs self-intersecting and invalid polygons.
# Works on all GEOS versions -- no GEOS 3.10 requirement.
# Removes anything that is not a polygon or multipolygon after repair,
# as mixed-dimension geometries cause overlay() to fail.
# =============================================================================

def clean_geodataframe(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """
    Repair invalid geometries and remove non-polygon features.

    buffer(0) is the standard fix for self-intersecting polygons and works
    on all GEOS versions. Anything that cannot be repaired is dropped.

    Args:
        gdf: Input GeoDataFrame.

    Returns:
        Cleaned GeoDataFrame with only valid polygon/multipolygon geometries.
    """
    def fix_geom(g):
        if g is None:
            return None
        try:
            # buffer(0) repairs most invalid polygons without needing GEOS 3.10
            fixed = g.buffer(0)
            if isinstance(fixed, (Polygon, MultiPolygon)) and not fixed.is_empty:
                return fixed
            return None
        except Exception:
            return None

    gdf = gdf.copy()
    gdf["geometry"] = gdf["geometry"].apply(fix_geom)

    # Drop rows where geometry could not be repaired
    before = len(gdf)
    gdf    = gdf[gdf["geometry"].notna()].reset_index(drop=True)
    after  = len(gdf)

    if before != after:
        print(f"  Dropped {before - after} invalid geometries.")

    return gdf




In [0]:
# =============================================================================
# GEOMETRY CLEANING
# buffer(0) repairs self-intersecting and invalid polygons.
# Works on all GEOS versions -- no GEOS 3.10 requirement.
# Removes anything that is not a polygon or multipolygon after repair,
# as mixed-dimension geometries cause overlay() to fail.
# =============================================================================
def clean_geodataframe(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """
    Repair invalid geometries using buffer(0).
    Does not filter by geometry type -- just fixes invalidity.
    """
    def fix_geom(g):
        if g is None:
            return None
        try:
            if g.is_valid:
                return g          # Already valid -- no repair needed
            return g.buffer(0)    # Repair invalid geometry
        except Exception:
            return g              # Return original if repair fails

    gdf = gdf.copy()
    gdf["geometry"] = gdf["geometry"].apply(fix_geom)
    gdf = gdf[gdf["geometry"].notna()].reset_index(drop=True)
    return gdf


In [0]:
# =============================================================================
# LOAD REFERENCE TABLES FROM DELTA INTO PANDAS
# We collect() to bring the data to the driver for GeoPandas processing.
# This is safe at these data volumes.
# =============================================================================

print("Loading EA flood areas from Delta...")

# Load FWAs and FAAs and combine into one DataFrame with an area_type column.
fwa_pdf = spark.table(TBL_EA_FWA).toPandas()
faa_pdf = spark.table(TBL_EA_FAA).toPandas()
ea_pdf  = pd.concat([fwa_pdf, faa_pdf], ignore_index=True)

print(f"  {len(ea_pdf)} EA flood areas loaded.")

print("Loading constituency boundaries from Delta...")
const_pdf = spark.table(TBL_CONSTITUENCIES).toPandas()
print(f"  {len(const_pdf)} constituencies loaded.")

In [0]:
# =============================================================================
# CONVERT TO GEODATAFRAMES
# GeoJSON strings stored in Delta are parsed back to Shapely geometry objects.
# =============================================================================

def geojson_str_to_geodataframe(pdf: pd.DataFrame, crs: str = "EPSG:4326") -> gpd.GeoDataFrame:
    """
    Convert a Pandas DataFrame with a GeoJSON geometry string column
    into a GeoPandas GeoDataFrame.
    """
    geometries = pdf["geometry"].apply(lambda s: shape(json.loads(s)) if s else None)
    return gpd.GeoDataFrame(pdf, geometry=geometries, crs=crs)


ea_gdf    = geojson_str_to_geodataframe(ea_pdf)
const_gdf = geojson_str_to_geodataframe(const_pdf)


In [0]:
# =============================================================================
# PROJECT TO BRITISH NATIONAL GRID
# EPSG:4326 coordinates are in degrees -- area calculations are inaccurate.
# EPSG:27700 (BNG) measures in metres, giving accurate area calculations.
# =============================================================================

print("Projecting to British National Grid (EPSG:27700)...")
ea_bng    = ea_gdf.to_crs("EPSG:27700")
const_bng = const_gdf.to_crs("EPSG:27700")


In [0]:
# =============================================================================
# CLEAN GEOMETRIES
# Must be done after projection -- reprojection can introduce small invalidity.
# =============================================================================

print("Cleaning EA geometries...")
ea_bng = clean_geodataframe(ea_bng)

print("Cleaning constituency geometries...")
const_bng = clean_geodataframe(const_bng)

print(f"After cleaning: {len(ea_bng)} EA areas, {len(const_bng)} constituencies.")


In [0]:
# =============================================================================
# SPATIAL JOIN
# sjoin finds all EA area / constituency pairs that intersect.
# Much cheaper than overlay -- finds candidate pairs without computing
# intersection geometry. We compute intersection area pair by pair below.
# =============================================================================

print("Running spatial join to find intersecting pairs...")

joined = gpd.sjoin(
    ea_bng,
    const_bng[["constituency_id", "name", "geometry"]],
    how="inner",
    predicate="intersects"
)

print(f"  Found {len(joined)} intersecting pairs.")


In [0]:
# =============================================================================
# COMPUTE INTERSECTION PERCENTAGES PAIR BY PAIR
# For each matched pair, compute the intersection geometry and its area.
# Done one pair at a time to keep memory usage flat -- gpd.overlay() on the
# full dataset causes an OOM kill (exit code 137).
#
# intersection_pct = intersection area / EA area
# Tells us what proportion of the EA flood area falls within the constituency.
# =============================================================================

print("Computing intersection percentages pair by pair...")

# Build a lookup from constituency_id to geometry for fast access.
const_geom_lookup = dict(
    zip(const_bng["constituency_id"], const_bng["geometry"])
)

lookup_rows = []

for i, (idx, row) in enumerate(joined.iterrows()):

    if i % 500 == 0:
        print(f"  Processing pair {i + 1}/{len(joined)}...")

    ea_geom    = ea_bng.loc[idx, "geometry"]
    const_geom = const_geom_lookup.get(row["constituency_id"])

    try:
        intersection_area = ea_geom.intersection(const_geom).area
        ea_area           = ea_geom.area
        pct               = intersection_area / ea_area if ea_area > 0 else 0.0
    except Exception:
        pct = 0.0

    lookup_rows.append(Row(
        ea_area_code      = str(row.get("ea_area_code")    or ""),
        ea_area_type      = str(row.get("ea_area_type")    or ""),
        ea_area_name      = str(row.get("ea_area_name")    or ""),
        type_code         = str(row.get("type_code")       or ""),
        type_text         = str(row.get("type_text")       or ""),
        constituency_id   = str(row.get("constituency_id") or ""),
        constituency_name = str(row.get("name")            or ""),
        intersection_pct  = float(pct),
        last_computed_at  = now_iso
    ))

print(f"Built {len(lookup_rows)} lookup rows.")


In [0]:
# =============================================================================
# WRITE TO DELTA
# Full overwrite -- this table is always recomputed from scratch.
# =============================================================================

if not lookup_rows:
    raise Exception("No lookup rows built -- check geometry cleaning output above.")

lookup_schema = StructType([
    StructField("ea_area_code",      StringType(), True),
    StructField("ea_area_type",      StringType(), True),
    StructField("ea_area_name",      StringType(), True),
    StructField("type_code",         StringType(), True),
    StructField("type_text",         StringType(), True),
    StructField("constituency_id",   StringType(), True),
    StructField("constituency_name", StringType(), True),
    StructField("intersection_pct",  FloatType(),  True),
    StructField("last_computed_at",  StringType(), True),
])

lookup_df = spark.createDataFrame(lookup_rows, schema=lookup_schema)

(
    lookup_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TBL_EA_CONST_LOOKUP)
)

written = spark.table(TBL_EA_CONST_LOOKUP).count()
print(f"Verified: {written} rows written to {TBL_EA_CONST_LOOKUP}.")

print("Lookup table computation complete.")
dbutils.notebook.exit("success")

